In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm
import statsmodels.api as sm

In [2]:
from sklearn.covariance import LedoitWolf

In [ ]:
def load_industry_data(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df / 100  # convert to decimal

def load_factor_data(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df / 100

def load_risk_free(factor_df):
    return factor_df['RF']

In [ ]:
def sample_mean(returns):
    return returns.mean()

def sample_covariance(returns):
    return returns.cov()

def excess_returns(returns, rf):
    return returns.sub(rf, axis=0)

In [ ]:
def bayes_stein_mean(returns):
    mu = returns.mean()
    sigma = returns.cov()
    T = len(returns)
    
    grand_mean = mu.mean()
    shrinkage_intensity = ((len(mu) + 2) / 
                          ((mu - grand_mean).T @ 
                           np.linalg.inv(sigma) @ 
                           (mu - grand_mean)))
    
    mu_bs = (1 - shrinkage_intensity) * mu + shrinkage_intensity * grand_mean
    return mu_bs

In [ ]:
def shrink_covariance(returns):
    lw = LedoitWolf()
    lw.fit(returns)
    return pd.DataFrame(lw.covariance_, 
                        index=returns.columns, 
                        columns=returns.columns)

In [ ]:
def portfolio_return(weights, mean_returns):
    return weights @ mean_returns

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(weights.T @ cov_matrix @ weights)

In [ ]:
def global_minimum_variance(cov_matrix):
    n = len(cov_matrix)
    ones = np.ones(n)
    
    inv_cov = np.linalg.inv(cov_matrix)
    weights = inv_cov @ ones / (ones.T @ inv_cov @ ones)
    return weights

In [ ]:
def tangency_portfolio(mean_returns, cov_matrix, rf):
    excess = mean_returns - rf
    inv_cov = np.linalg.inv(cov_matrix)
    
    weights = inv_cov @ excess / (np.ones(len(mean_returns)).T @ inv_cov @ excess)
    return weights

In [ ]:
def efficient_frontier(mean_returns, cov_matrix, num_points=100):
    target_returns = np.linspace(min(mean_returns), 
                                 max(mean_returns), 
                                 num_points)
    
    frontier_vol = []
    
    for target in target_returns:
        constraints = (
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w: w @ mean_returns - target}
        )
        
        result = minimize(portfolio_volatility,
                          x0=np.ones(len(mean_returns))/len(mean_returns),
                          args=(cov_matrix,),
                          constraints=constraints)
        
        frontier_vol.append(result.fun)
    
    return target_returns, frontier_vol

In [ ]:
def sharpe_ratio(mean_return, volatility, rf):
    return (mean_return - rf) / volatility

def portfolio_stats(weights, returns, rf):
    port_returns = returns @ weights
    mean = port_returns.mean()
    vol = port_returns.std()
    sr = sharpe_ratio(mean, vol, rf)
    
    return mean, vol, sr

In [ ]:
def jobson_korkie(sr1, sr2, T):
    diff = sr1 - sr2
    var = (1/T) * (1 + 0.5 * (sr1**2 + sr2**2))
    z = diff / np.sqrt(var)
    p_value = 2 * (1 - norm.cdf(abs(z)))
    return z, p_value

In [ ]:
def factor_frontier(factor_returns):
    mu = sample_mean(factor_returns)
    cov = sample_covariance(factor_returns)
    return efficient_frontier(mu, cov)

In [ ]:
def compute_alpha(fund_returns, factor_returns):
    X = sm.add_constant(factor_returns)
    model = sm.OLS(fund_returns, X).fit()
    return model.params[0]  

In [ ]:
def plot_frontiers(frontiers_dict):
    plt.figure(figsize=(10,7))
    
    for label, (ret, vol) in frontiers_dict.items():
        plt.plot(vol, ret, label=label)
    
    plt.xlabel("Volatility")
    plt.ylabel("Expected Return")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
train = returns.loc[:'2002-12']
test  = returns.loc['2003-01':]

mu_train = sample_mean(train)
cov_train = sample_covariance(train)

weights_tan = tangency_portfolio(mu_train, cov_train, rf_train)

mean_test, vol_test, sr_test = portfolio_stats(weights_tan, test, rf_test)